# Experiment Results Analysis 

The object of this notebook is to conduct the final experiment analysis in order to test hypothesis regarding algorithm performance.

In this notebook the different executed batches of experiments will be consolidated in a unique database to analyze its results.


## Analysis Conducted

1. Compare AUC for each algorithm in each enviroment (Comparisons are conducted across algorithms in the same environment)

2. Compare the Mean Best So Far Curve in each enviroment (Comparisons are conducted across algorithms in the same environment) with confidence intervals

3. Perform the wilcoxon Rank Sum test in each enviroment for selected pairwise comaprisons (Comparisons are conducted across algorithms in the same environment) with confidence intervals


## Algorithms to compare

1. Binary Particle Swarm Optimization - (PSO)

2. Binary Global Random Search - (Random)

3. Standard Binary Representation GA (Generic)

4. Standard Mixed Integer Representation GA (mixed_generic)

5. Macro_Micro CX Operator GA - (macro_micro)

6. Macro CX Operator GA - Deactivates Micro CX operator - (micro)

7. Micro CX Operator GA - Deactivates Macro CX operator - (macro)

8. Parents CX Operator - Uses a parent recombination CX Operator (recomb)

## 1. Library Imports

In [191]:
import pandas as pd 
import numpy as np
from scipy.stats import mannwhitneyu
import matplotlib.pyplot as plt
import seaborn as sns
import os 
import ast
from typing import List, Dict 
import re

## 2. Data Import

In [192]:
# Check wd
print(os.getcwd())



c:\Users\andre\Repositories\FTZ_model_2.0\final_results\experiment


In [193]:
def load_and_concat_csv(paths: List[str], *, ignore_index: bool = True) -> pd.DataFrame:
    """
    Load multiple CSV files and concatenate them into a single DataFrame.

    Parameters
    ----------
    paths : List[str]
        List of file paths to CSV files.
    ignore_index : bool, optional
        If True, reset the index in the concatenated DataFrame. Default is True.

    Returns
    -------
    pd.DataFrame
        Concatenated DataFrame containing all CSV files.
    """
    dataframes = []
    for path in paths:
        try:
            df = pd.read_csv(path)
            df["source_file"] = path  # optional: keep track of origin
            dataframes.append(df)
            print(f"Loaded: {path} ({len(df)} rows)")
        except FileNotFoundError:
            print(f" File not found: {path}")
        except pd.errors.EmptyDataError:
            print(f" Empty file skipped: {path}")
        except Exception as e:
            print(f" Error reading {path}: {e}")

    if not dataframes:
        raise ValueError("No valid CSV files loaded.")

    combined_df = pd.concat(dataframes, ignore_index=ignore_index)
    print(f" Combined DataFrame: {combined_df.shape[0]} rows, {combined_df.shape[1]} columns")

    return combined_df

In [194]:
paths = [
    r"data_exp\experiment_final_macrmicro_mixedgeneric\experiment_final_macrmicro_mixedgeneric\results.csv",
    r"data_exp\experiment_final_nocross_jmicro_jmacro_recomb\experiment_final_nocross_jmicro_jmacro_recomb\results.csv",
    r"data_exp\final_experiment_rand_pso_gen (1)\final_experiment_rand_pso_gen\results.csv"
]


In [195]:
df_all = load_and_concat_csv(paths)

Loaded: data_exp\experiment_final_macrmicro_mixedgeneric\experiment_final_macrmicro_mixedgeneric\results.csv (32480 rows)
Loaded: data_exp\experiment_final_nocross_jmicro_jmacro_recomb\experiment_final_nocross_jmicro_jmacro_recomb\results.csv (64960 rows)
Loaded: data_exp\final_experiment_rand_pso_gen (1)\final_experiment_rand_pso_gen\results.csv (48720 rows)
 Combined DataFrame: 146160 rows, 11 columns


## 3. Clean Data 

Due to a code / paralleization mistake, some rows are duplicated

In [196]:
df_all.head()

,env_name,run,algorithm,seed,best_curve,best_value,best_genome,all_best_genomes,elapsed_sec,success,source_file
0,Linear_standard,1,macro_micro,483575,"[38.9, 39.05, 39.05, 39.2, 39.35, 39.35, 39.35...",39.35,"[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1]","[[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...",81.426382,True,data_exp\experiment_final_macrmicro_mixedgener...
1,Linear_standard,11,macro_micro,1424007,"[38.75, 38.9, 38.9, 39.05, 39.05, 39.2, 39.2, ...",39.35,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1]","[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...",81.670461,True,data_exp\experiment_final_macrmicro_mixedgener...
2,Linear_standard,13,macro_micro,1242588,"[38.75, 38.9, 39.05, 39.05, 39.2, 39.2, 39.2, ...",39.35,"[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1]","[[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...",81.777966,True,data_exp\experiment_final_macrmicro_mixedgener...
3,Linear_standard,10,macro_micro,874627,"[39.05, 39.05, 39.05, 39.2, 39.2, 39.2, 39.35,...",39.35,"[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1]","[[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...",81.887681,True,data_exp\experiment_final_macrmicro_mixedgener...
4,Linear_standard,6,macro_micro,1081112,"[38.75, 38.9, 39.05, 39.05, 39.35, 39.35, 39.3...",39.35,"[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1]","[[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...",82.319809,True,data_exp\experiment_final_macrmicro_mixedgener...


In [197]:
df_all.columns

Index(['env_name', 'run', 'algorithm', 'seed', 'best_curve', 'best_value',
       'best_genome', 'all_best_genomes', 'elapsed_sec', 'success',
       'source_file'],
      dtype='object')

In [46]:
# Keep the row with the smallest elapsed_sec per (env_name, algorithm, seed, run)
before = len(df_all)
print(f"Total rows before removing duplicates: {before}")

df_tmp = df_all.copy()
df_tmp["elapsed_sec"] = pd.to_numeric(df_tmp.get("elapsed_sec", np.nan), errors="coerce")
df_tmp["_elapsed_sort"] = df_tmp["elapsed_sec"].fillna(np.inf)

df_analysis = (
    df_tmp
    .sort_values(["env_name", "algorithm", "seed", "run", "_elapsed_sort"],
                 ascending=[True, True, True, True, True])
    .drop_duplicates(subset=["env_name", "algorithm", "seed", "run"], keep="first")
    .drop(columns=["_elapsed_sort"])
    .reset_index(drop=True)
)

after = len(df_analysis)
print(f"Removed {before - after} duplicate rows")


Total rows before removing duplicates: 146160
Removed 136080 duplicate rows


In [199]:
df_analysis.shape

(10181, 11)

In [200]:
df_analysis.head()

,env_name,run,algorithm,seed,best_curve,best_value,best_genome,all_best_genomes,elapsed_sec,success,source_file
0,Linear_standard,1,macro_micro,483575,"[38.9, 39.05, 39.05, 39.2, 39.35, 39.35, 39.35...",39.35,"[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1]","[[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...",81.426382,True,data_exp\experiment_final_macrmicro_mixedgener...
1,Linear_standard,11,macro_micro,1424007,"[38.75, 38.9, 38.9, 39.05, 39.05, 39.2, 39.2, ...",39.35,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1]","[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...",81.670461,True,data_exp\experiment_final_macrmicro_mixedgener...
2,Linear_standard,13,macro_micro,1242588,"[38.75, 38.9, 39.05, 39.05, 39.2, 39.2, 39.2, ...",39.35,"[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1]","[[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...",81.777966,True,data_exp\experiment_final_macrmicro_mixedgener...
3,Linear_standard,10,macro_micro,874627,"[39.05, 39.05, 39.05, 39.2, 39.2, 39.2, 39.35,...",39.35,"[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1]","[[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...",81.887681,True,data_exp\experiment_final_macrmicro_mixedgener...
4,Linear_standard,6,macro_micro,1081112,"[38.75, 38.9, 39.05, 39.05, 39.35, 39.35, 39.3...",39.35,"[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1]","[[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...",82.319809,True,data_exp\experiment_final_macrmicro_mixedgener...


## 4. Graphics of Mean Best So Far Curves

In [201]:
df_graphics = df_analysis[["env_name", "run", "seed", "algorithm", "best_curve", "best_value"]].copy()


In [202]:
df_graphics.head()

,env_name,run,seed,algorithm,best_curve,best_value
0,Linear_standard,1,483575,macro_micro,"[38.9, 39.05, 39.05, 39.2, 39.35, 39.35, 39.35...",39.35
1,Linear_standard,11,1424007,macro_micro,"[38.75, 38.9, 38.9, 39.05, 39.05, 39.2, 39.2, ...",39.35
2,Linear_standard,13,1242588,macro_micro,"[38.75, 38.9, 39.05, 39.05, 39.2, 39.2, 39.2, ...",39.35
3,Linear_standard,10,874627,macro_micro,"[39.05, 39.05, 39.05, 39.2, 39.2, 39.2, 39.35,...",39.35
4,Linear_standard,6,1081112,macro_micro,"[38.75, 38.9, 39.05, 39.05, 39.35, 39.35, 39.3...",39.35


In [203]:
df_graphics["best_curve"].dtype


dtype('O')

In [204]:
def safe_parse_with_inf(x):
    if not isinstance(x, str):
        return np.array(x, dtype=float) if isinstance(x, (list, np.ndarray)) else np.array([])

    # Replace 'inf', '-inf', 'nan' with strings that eval can interpret
    x = re.sub(r'\b-inf\b', 'float("-inf")', x)
    x = re.sub(r'\binf\b', 'float("inf")', x)
    x = re.sub(r'\bnan\b', 'float("nan")', x)

    try:
        parsed = ast.literal_eval(x)
        return np.array(parsed, dtype=float)
    except Exception:
        return np.array([])

df_graphics["best_curve"] = df_graphics["best_curve"].apply(safe_parse_with_inf)

In [205]:
df_graphics.columns

Index(['env_name', 'run', 'seed', 'algorithm', 'best_curve', 'best_value'], dtype='object')

### Graphics 

In [206]:
def clean_curve(arr):
    arr = np.array(arr, dtype=float)
    mask_invalid = ~np.isfinite(arr)  # detects inf, -inf, nan
    n_invalid = np.sum(mask_invalid)
    arr = arr[np.isfinite(arr)]       # keep only finite values
    return arr, n_invalid

# Clean curves and count removals
invalid_counts = []
cleaned_curves = []

for idx, arr in enumerate(df_graphics["best_curve"]):
    cleaned, n_invalid = clean_curve(arr)
    invalid_counts.append(n_invalid)
    cleaned_curves.append(cleaned)

df_graphics["best_curve"] = cleaned_curves
df_graphics["invalid_points"] = invalid_counts

print(f"Total invalid values removed: {sum(invalid_counts)}")

Total invalid values removed: 0


In [207]:
from scipy.stats import t

from scipy.stats import t
import numpy as np

def mean_curve_with_ci(curves, debug=False):
    """
    Computes the mean curve and 95% confidence interval across runs.
    Filters out invalid, empty, or malformed curves.
    """

    clean_curves = []
    for i, c in enumerate(curves):
        # Skip non-list/array items or empty ones
        if not isinstance(c, (list, np.ndarray)):
            if debug:
                print(f"⚠️ Run {i} skipped (not list/array): {type(c)} -> {c}")
            continue
        c = np.array(c, dtype=float).flatten()

        # Skip empty or NaN-only curves
        if len(c) == 0 or not np.isfinite(np.sum(c)):
            if debug:
                print(f"⚠️ Run {i} skipped (empty or invalid values): {c}")
            continue

        clean_curves.append(c)

    # If none valid, return empty arrays
    if len(clean_curves) == 0:
        if debug:
            print("❌ No valid curves in group.")
        return np.array([]), np.array([]), np.array([])

    # Truncate to the shortest length to align
    min_len = min(len(c) for c in clean_curves)
    if debug:
        print(f"✅ Using {len(clean_curves)} valid curves, truncated to length {min_len}")

    data = np.stack([c[:min_len] for c in clean_curves], axis=0)

    # Compute statistics
    mean = np.nanmean(data, axis=0)
    std = np.nanstd(data, axis=0, ddof=1)
    n = data.shape[0]
    alpha = 0.05
    tcrit = t.ppf(1 - alpha/2, df=n - 1) if n > 1 else 0
    se = std / np.sqrt(n)
    ci = tcrit * se

    return mean, mean - ci, mean + ci


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import os

# === Output folder on your Desktop ===
desktop = os.path.join(os.path.expanduser("~"), "Desktop")
output_dir = os.path.join(desktop, "FTZ_Plots")
os.makedirs(output_dir, exist_ok=True)
print(f" Plots will be saved in: {output_dir}\n")

# === Group by environment and algorithm ===
grouped = df_graphics.groupby(["env_name", "algorithm"])

# Unique environments
envs = df_graphics["env_name"].unique()

#  Print how many and which environments will be plotted
print(f" Plotting {len(envs)} environments:")
for i, e in enumerate(envs, 1):
    print(f"   {i}. {e}")

# === Plot per environment ===
for env in envs:
    subdf_env = df_graphics[df_graphics["env_name"] == env]
    fig, ax = plt.subplots(figsize=(8, 5))

    for algo, df_algo in subdf_env.groupby("algorithm"):
        curves = df_algo["best_curve"].tolist()
        mean, lower, upper = mean_curve_with_ci(curves, debug=False)
        if len(mean) == 0:
            continue
        x = np.arange(len(mean))
        ax.plot(x, mean, label=f"{algo}")
        ax.fill_between(x, lower, upper, alpha=0.2)

    ax.set_title(f"Environment: {env}")
    ax.set_xlabel("Step / Generation")
    ax.set_ylabel("Average Best Curve")
    ax.legend()
    plt.tight_layout()

    # === Save the plot ===
    safe_name = "".join(c if c.isalnum() or c in ('_', '-') else "_" for c in env)
    png_path = os.path.join(output_dir, f"{safe_name}.png")

    plt.savefig(pdf_path, bbox_inches="tight")
    print(f"✅ Saved: {png_path}")
    plt.close(fig)

print("\n🎨 All environment plots saved successfully!")


 Plots will be saved in: C:\Users\andre\Desktop\FTZ_Plots

 Plotting 28 environments:
   1. Linear_standard
   2. Linear_perturbed
   3. Linear All-Inputs_standard
   4. Linear All-Inputs_perturbed
   5. Assembly Tree_standard
   6. Assembly Tree_perturbed
   7. Tri-Spine Convergent_standard
   8. Tri-Spine Convergent_perturbed
   9. Sequential Mergers with External Inputs_standard
   10. Sequential Mergers with External Inputs_perturbed
   11. Butterfly_standard
   12. Butterfly_perturbed
   13. Tri-Cluster_standard
   14. Tri-Cluster_perturbed
   15. All Inputs_standard
   16. All Inputs_perturbed
   17. Mangrove_standard
   18. Mangrove_perturbed
   19. Dominant Input_standard
   20. Dominant Input_perturbed
   21. Successive Diamonds_standard
   22. Successive Diamonds_perturbed
   23. Interlaced Pathway_standard
   24. Interlaced Pathway_perturbed
   25. Explosive Expansion_standard
   26. Explosive Expansion_perturbed
   27. Mixed Test Graph_standard
   28. Mixed Test Graph_pertu

✅ Saved: C:\Users\andre\Desktop\FTZ_Plots\Linear_standard.png
✅ Saved: C:\Users\andre\Desktop\FTZ_Plots\Linear_perturbed.png
✅ Saved: C:\Users\andre\Desktop\FTZ_Plots\Linear_All-Inputs_standard.png
✅ Saved: C:\Users\andre\Desktop\FTZ_Plots\Linear_All-Inputs_perturbed.png
✅ Saved: C:\Users\andre\Desktop\FTZ_Plots\Assembly_Tree_standard.png
✅ Saved: C:\Users\andre\Desktop\FTZ_Plots\Assembly_Tree_perturbed.png
✅ Saved: C:\Users\andre\Desktop\FTZ_Plots\Tri-Spine_Convergent_standard.png
✅ Saved: C:\Users\andre\Desktop\FTZ_Plots\Tri-Spine_Convergent_perturbed.png
✅ Saved: C:\Users\andre\Desktop\FTZ_Plots\Sequential_Mergers_with_External_Inputs_standard.png
✅ Saved: C:\Users\andre\Desktop\FTZ_Plots\Sequential_Mergers_with_External_Inputs_perturbed.png
✅ Saved: C:\Users\andre\Desktop\FTZ_Plots\Butterfly_standard.png
✅ Saved: C:\Users\andre\Desktop\FTZ_Plots\Butterfly_perturbed.png
✅ Saved: C:\Users\andre\Desktop\FTZ_Plots\Tri-Cluster_standard.png
✅ Saved: C:\Users\andre\Desktop\FTZ_Plots\Tri-C

In [214]:
df_graphics['env_name'].unique()

array(['Linear_standard', 'Linear_perturbed',
       'Linear All-Inputs_standard', 'Linear All-Inputs_perturbed',
       'Assembly Tree_standard', 'Assembly Tree_perturbed',
       'Tri-Spine Convergent_standard', 'Tri-Spine Convergent_perturbed',
       'Sequential Mergers with External Inputs_standard',
       'Sequential Mergers with External Inputs_perturbed',
       'Butterfly_standard', 'Butterfly_perturbed',
       'Tri-Cluster_standard', 'Tri-Cluster_perturbed',
       'All Inputs_standard', 'All Inputs_perturbed', 'Mangrove_standard',
       'Mangrove_perturbed', 'Dominant Input_standard',
       'Dominant Input_perturbed', 'Successive Diamonds_standard',
       'Successive Diamonds_perturbed', 'Interlaced Pathway_standard',
       'Interlaced Pathway_perturbed', 'Explosive Expansion_standard',
       'Explosive Expansion_perturbed', 'Mixed Test Graph_standard',
       'Mixed Test Graph_perturbed'], dtype=object)

## 5. Hypothesis testing

In [209]:
df_wilcoxon = df_graphics[["env_name", "algorithm", "best_value"]].copy()

df_wilcoxon.head()

,env_name,algorithm,best_value
0,Linear_standard,macro_micro,39.35
1,Linear_standard,macro_micro,39.35
2,Linear_standard,macro_micro,39.35
3,Linear_standard,macro_micro,39.35
4,Linear_standard,macro_micro,39.35


In [215]:
pairs = [
    # macro_micro vs others
    ("macro_micro", "mixed_generic"),
    ("mixed_generic", "macro_micro"),
    ("macro_micro", "generic"),
    ("generic", "macro_micro"),
    ("macro_micro", "macro"),
    ("macro", "macro_micro"),
    ("macro_micro", "micro"),
    ("micro", "macro_micro"),
    ('recomb','macro_micro'),
    ('macro_micro','recomb'),

    # mixed_generic vs others
    ("mixed_generic", "pso"),
    ("pso", "mixed_generic"),
    ("mixed_generic", "generic"),
    ("generic", "mixed_generic"),

    # pso vs others
    ("pso", "random"),
    ("random", "pso"),
]


alpha = 0.05  # significance level
results = []


In [216]:
print(f"\n Performing pairwise Mann–Whitney U tests at α={alpha}\n")

# Loop over each environment
for env in df_wilcoxon["env_name"].unique():
    df_env = df_wilcoxon[df_graphics["env_name"] == env]

    for a1, a2 in pairs:
        # Ensure both algorithms exist in this environment
        if a1 not in df_env["algorithm"].unique() or a2 not in df_env["algorithm"].unique():
            continue

        vals1 = df_env.loc[df_env["algorithm"] == a1, "best_value"].astype(float).values
        vals2 = df_env.loc[df_env["algorithm"] == a2, "best_value"].astype(float).values

        # Skip if too few observations
        if len(vals1) < 3 or len(vals2) < 3:
            continue

        # Mann–Whitney test (one-sided): H₁ = a1 > a2
        stat, p = mannwhitneyu(vals1, vals2, alternative="greater")
        better = p < alpha

        results.append({
            "env_name": env,
            "algorithm_1": a1,
            "algorithm_2": a2,
            "p_value": p,
            "reject_H0_(a1> a2)": better
        })

# Create DataFrame of all tests
df_mannwhitney = pd.DataFrame(results)

# Summarize: count environments where a1 significantly better
summary = (
    df_mannwhitney.groupby(["algorithm_1", "algorithm_2"])["reject_H0_(a1> a2)"]
    .sum()
    .reset_index()
    .rename(columns={"reject_H0_(a1> a2)": "envs_where_a1_was_significantly_better"})
)

# Print detailed results and summary
print("\n📄 Detailed per-environment results:")
print(df_mannwhitney.to_string(index=False))

print("\n📊 Summary across environments:")
print(summary.to_string(index=False))



 Performing pairwise Mann–Whitney U tests at α=0.05


📄 Detailed per-environment results:
                                         env_name   algorithm_1   algorithm_2      p_value  reject_H0_(a1> a2)
                                  Linear_standard   macro_micro mixed_generic 1.647801e-01               False
                                  Linear_standard mixed_generic   macro_micro 8.473184e-01               False
                                  Linear_standard   macro_micro       generic 1.000000e+00               False
                                  Linear_standard       generic   macro_micro 1.000000e+00               False
                                  Linear_standard   macro_micro         macro 1.000000e+00               False
                                  Linear_standard         macro   macro_micro 1.000000e+00               False
                                  Linear_standard   macro_micro         micro 1.000000e+00               False
                     

In [ ]:
# Hello